In [1]:
import os
import pandas as pd
import nltk
import spacy
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from gensim.models import FastText

In [2]:
os.chdir("../")

In [4]:
from src.functions import remove_stopwords_punctuation, remove_outliers, lematiza_tokens, embedding_ft, salva_embeddings

In [5]:
df = pd.read_csv("./data/buscape.csv")

In [6]:
df

,original_index,review_text,review_text_processed,review_text_tokenized,polarity,rating,kfold_polarity,kfold_rating
0,4_55516,"Estou muito satisfeito, o visor é melhor do qu...","estou muito satisfeito, o visor e melhor do qu...","['estou', 'muito', 'satisfeito', 'visor', 'mel...",1.0,4,1,1
1,minus_1_105339,"""muito boa\n\nO que gostei: preco\n\nO que não...","""muito boa\n\no que gostei: preco\n\no que nao...","['muito', 'boa', 'que', 'gostei', 'preco', 'qu...",1.0,5,1,1
2,23_382139,"Rápida, ótima qualidade de impressão e fácil d...","rapida, otima qualidade de impressao e facil d...","['rapida', 'otima', 'qualidade', 'de', 'impres...",1.0,5,1,1
3,2_446456,Produto de ótima qualidade em todos os quesito!,produto de otima qualidade em todos os quesito!,"['produto', 'de', 'otima', 'qualidade', 'em', ...",1.0,5,1,1
4,0_11324,Precisava comprar uma tv compatível com meu dv...,precisava comprar uma tv compativel com meu dv...,"['precisava', 'comprar', 'uma', 'tv', 'compati...",1.0,5,1,1
...,...,...,...,...,...,...,...,...
84986,1_422965,"Produto muito bom, simples e barato","produto muito bom, simples e barato","['produto', 'muito', 'bom', 'simples', 'barato']",1.0,5,10,10
84987,minus_1_150466,O esquema antigo de desmontagem e limpeza das ...,o esquema antigo de desmontagem e limpeza das ...,"['esquema', 'antigo', 'de', 'desmontagem', 'li...",NaN,3,-1,10
84988,0_414799,Esse jogo é muito maneiro é um jogo onde vc te...,esse jogo e muito maneiro e um jogo onde vc te...,"['esse', 'jogo', 'muito', 'maneiro', 'um', 'jo...",1.0,5,10,10
84989,0_389898,Muito bom e intuitivo!\n\nO que gostei: Educa ...,muito bom e intuitivo!\n\no que gostei: educa ...,"['muito', 'bom', 'intuitivo', 'que', 'gostei',...",NaN,3,-1,10


### 1. Train-Test Split

In [7]:
X_data = df.drop(columns=['polarity'])
y_data = df['polarity']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.3, random_state=42)

## Conjunto de Treino

### 2. Limpeza de Dados
##### 2.1. Tratamento de Nulos

In [9]:
X_train.shape

(59493, 7)

In [10]:
X_train['review_text'].isnull().sum()

np.int64(1)

In [11]:
X_train = X_train.dropna(subset=['review_text'])

In [12]:
X_train.shape

(59492, 7)

In [13]:
# paridade de índices
y_train = y_train.loc[X_train.index]

In [14]:
y_train.shape

(59492,)

##### 2.2. Tratamento de Outliers

In [15]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [16]:
X_train_token = pd.DataFrame()
X_train_token['token'] = X_train['review_text']
X_train_token

,token
69382,Ótimo aparelho. Para quem já usa ou irá experi...
11241,"Apesar de o acesso ser USB 2.0, tem boa veloci..."
80248,comparado ao produto que usava anteriormente é...
61220,Muito bom\n\nO que gostei: Muito instrutivo\n\...
76452,Produto excelente. Sua qualidade de imagem cha...
...,...
6265,acho lindo quero esse modelo de qualquer geito...
54886,todomundo vai querer um ipad como esse
76820,"foi um investimento muito bom, sem arrependime..."
860,"Considero um bom aparelho, com um conceito eco..."


In [17]:
X_train_token['token'] = X_train_token['token'].apply(word_tokenize)

In [18]:
X_train_token['token'] = X_train_token['token'].apply(lambda text: remove_stopwords_punctuation(text))

In [19]:
X_train_token['len_token'] = X_train_token['token'].apply(lambda text: len(text))

In [20]:
X_train_token

,token,len_token
69382,"[Ótimo, aparelho, Para, usa, irá, experimentar...",25
11241,"[Apesar, acesso, USB, 2.0, boa, velocidade, tr...",105
80248,"[comparado, produto, usava, anteriormente, sup...",40
61220,"[Muito, bom, O, gostei, Muito, instrutivo, O, ...",10
76452,"[Produto, excelente, Sua, qualidade, imagem, c...",18
...,...,...
6265,"[acho, lindo, quero, modelo, qualquer, geito, ...",19
54886,"[todomundo, vai, querer, ipad]",4
76820,"[investimento, bom, arrependimento, O, gostei,...",25
860,"[Considero, bom, aparelho, conceito, ecológico...",25


In [21]:
X_train_token = remove_outliers(X_train_token, 'len_token')

In [22]:
X_train_token

,token,len_token
69382,"[Ótimo, aparelho, Para, usa, irá, experimentar...",25
80248,"[comparado, produto, usava, anteriormente, sup...",40
61220,"[Muito, bom, O, gostei, Muito, instrutivo, O, ...",10
76452,"[Produto, excelente, Sua, qualidade, imagem, c...",18
51615,"[MDesign, moderno, todas, tecnologias, necessa...",21
...,...,...
6265,"[acho, lindo, quero, modelo, qualquer, geito, ...",19
54886,"[todomundo, vai, querer, ipad]",4
76820,"[investimento, bom, arrependimento, O, gostei,...",25
860,"[Considero, bom, aparelho, conceito, ecológico...",25


In [23]:
# paridade de índices em X e y
X_train = X_train.loc[X_train_token.index]

In [24]:
y_train = y_train.loc[X_train_token.index]

In [25]:
print(X_train.shape, y_train.shape)

(55151, 7) (55151,)


### 3. Transformação
##### 3.1. Remoção de colunas "inúteis"

In [26]:
X_train.columns

Index(['original_index', 'review_text', 'review_text_processed',
       'review_text_tokenized', 'rating', 'kfold_polarity', 'kfold_rating'],
      dtype='str')

In [27]:
X_train = X_train.drop(columns=['original_index', 'review_text_processed', 'review_text_tokenized', 'rating', 'kfold_polarity', 'kfold_rating'])

In [28]:
print(f'colunas: {X_train.columns}. tipo: {type(X_train)}')

colunas: Index(['review_text'], dtype='str'). tipo: <class 'pandas.DataFrame'>


##### 3.2. Tratamento na label

- Transforma coluna binária em ternária

In [29]:
y_train.value_counts()

polarity
1.0    44023
0.0     3930
Name: count, dtype: int64

In [30]:
y_train.isnull().sum()

np.int64(7198)

In [31]:
y_train = y_train.map({1.0: 2, 0.0: 0})  # {valor_antigo: valor_atual}
y_train = y_train.fillna(1)

In [32]:
y_train.value_counts()

polarity
2.0    44023
1.0     7198
0.0     3930
Name: count, dtype: int64

##### 3.3. Resampling

In [33]:
type(y_train)

pandas.Series

In [34]:
us = RandomUnderSampler(random_state=0)

In [35]:
X_train, y_train = us.fit_resample(X_train, y_train)

In [36]:
print(X_train.shape, y_train.shape, type(X_train), type(y_train))

(11790, 1) (11790,) <class 'pandas.DataFrame'> <class 'pandas.Series'>


In [37]:
y_train.value_counts()

polarity
0.0    3930
1.0    3930
2.0    3930
Name: count, dtype: int64

##### 3.4. PROCESSAMENTO DE LINGUAGEM NATURAL
##### 3.4.1. Tokenização

In [38]:
X_train

,review_text
44756,Comprei o produto faz duas semanas mas só agor...
67481,"Nao compra, som muito ruim e pessima qualidade..."
108,"Péssimo, tem um barulho de cigarra que depois ..."
69342,"Produto inovador com a cara da Apple, acredito..."
80483,Deixa a desejar pelo preço do produto. Possui ...
...,...
56093,Ainda não tive. Mais já experimentei no da min...
41699,Amei o produto. Estou bastante satisfeita\n\nO...
49012,excelente\n\nO que gostei: praticidade em ter ...
17517,"Bom, como adquiri ontem rsrs a minha experiênc..."


In [39]:
X_train['review_text'] = X_train['review_text'].apply(word_tokenize)

In [40]:
X_train

,review_text
44756,"[Comprei, o, produto, faz, duas, semanas, mas,..."
67481,"[Nao, compra, ,, som, muito, ruim, e, pessima,..."
108,"[Péssimo, ,, tem, um, barulho, de, cigarra, qu..."
69342,"[Produto, inovador, com, a, cara, da, Apple, ,..."
80483,"[Deixa, a, desejar, pelo, preço, do, produto, ..."
...,...
56093,"[Ainda, não, tive, ., Mais, já, experimentei, ..."
41699,"[Amei, o, produto, ., Estou, bastante, satisfe..."
49012,"[excelente, O, que, gostei, :, praticidade, em..."
17517,"[Bom, ,, como, adquiri, ontem, rsrs, a, minha,..."


##### 3.4.2. Remoção de stopwords e pontuação

In [41]:
X_train['review_text'] = X_train['review_text'].apply(lambda x: remove_stopwords_punctuation(x))

In [42]:
X_train

,review_text
44756,"[Comprei, produto, faz, duas, semanas, agora, ..."
67481,"[Nao, compra, som, ruim, pessima, qualidade, i..."
108,"[Péssimo, barulho, cigarra, algum, tempom, uso..."
69342,"[Produto, inovador, cara, Apple, acredito, evo..."
80483,"[Deixa, desejar, preço, produto, Possui, opçõe..."
...,...
56093,"[Ainda, Mais, experimentei, amiga, ameei, Já, ..."
41699,"[Amei, produto, Estou, bastante, satisfeita, O..."
49012,"[excelente, O, gostei, praticidade, ter, unica..."
17517,"[Bom, adquiri, ontem, rsrs, experiência, tão, ..."


##### 3.4.3. Lemmatizing

In [43]:
!python -m spacy download pt_core_news_sm

     ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
      --------------------------------------- 0.3/13.0 MB ? eta -:--:--
     ------ --------------------------------- 2.1/13.0 MB 9.8 MB/s eta 0:00:02
     ---------------------- ----------------- 7.3/13.0 MB 16.2 MB/s eta 0:00:01
     ------------------------------------ -- 12.1/13.0 MB 18.0 MB/s eta 0:00:01
     --------------------------------------- 13.0/13.0 MB 15.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
nlp = spacy.load("pt_core_news_sm")

In [45]:
X_train["review_text"] = X_train["review_text"].apply(lambda x: lematiza_tokens(x, nlp))

In [46]:
X_train

,review_text
44756,"[Comprei, produto, fazer, dois, semana, agora,..."
67481,"[nao, compra, som, ruim, pessimo, qualidade, i..."
108,"[péssimo, barulho, cigarrar, algum, tempom, us..."
69342,"[produto, inovador, carar, Apple, acreditar, e..."
80483,"[Deixa, desejar, preço, produto, Possui, opção..."
...,...
56093,"[ainda, Mais, experimentar, amigo, ameei, já, ..."
41699,"[Amei, produto, estar, bastante, satisfeito, o..."
49012,"[excelente, o, gostar, praticidade, ter, unico..."
17517,"[bom, adquirir, ontem, rsrs, experiência, tão,..."


##### 3.4.4. Embedding

In [47]:
# FastText: representa a palavra pela soma dos n-gramas de caracteres, então consegue vetorizar palavras inéditas do conjunto de teste
model_embedding = FastText(X_train['review_text'], min_count=1, vector_size=100, window=5)

In [48]:
X_train['review_text'] = X_train['review_text'].apply(lambda x: embedding_ft(x, model_embedding))

In [49]:
X_train = X_train.rename(columns={'review_text': 'review_embedding'})

In [50]:
X_train

,review_embedding
44756,"[[0.6510803, 1.2826275, 0.44068852, 1.0763499,..."
67481,"[[0.565286, 0.64826435, 0.7957653, 1.1371483, ..."
108,"[[0.28849742, 0.896914, 1.1048805, 1.0288625, ..."
69342,"[[0.5020159, 1.3609866, 1.028918, 1.5520443, -..."
80483,"[[-0.15299532, 0.21269551, 0.2157306, 0.266025..."
...,...
56093,"[[0.28340092, 1.0862511, 0.91214573, 1.3662578..."
41699,"[[0.46518752, 0.6875162, 0.3214784, 0.50910157..."
49012,"[[-0.29945263, 1.016801, 1.0716661, 1.5223943,..."
17517,"[[-0.583551, 0.78032035, 1.3553377, 2.1414065,..."


### 4. Salvando os dados

In [51]:
salva_embeddings(X_train, y_train, 'data/X_train.pt', 'data/y_train.pt', vector_size=100)

c:\Users\roger\OneDrive\Documentos\Coding Repos\NLP-Sentiment-Classification-PyTorch\src\functions.py:65: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:255.)
  torch.tensor(row, dtype=torch.float32)


## Conjunto de Teste

O conjunto de teste recebe **só as transformações que o modelo precisa para conseguir ler o dado** — nada que altere a distribuição, porque ele tem que representar o mundo real na hora da avaliação.

| Etapa do treino | Aplicada no teste? | Motivo |
|---|---|---|
| Tratamento de nulos | SIM | `word_tokenize` quebra com `NaN` |
| Remoção de outliers (`len_token`) | NÃO | descartar reviews longas maquiaria a métrica |
| Remoção de colunas | SIM | o modelo espera as mesmas features |
| Mapeamento da label | SIM | os rótulos têm que significar o mesmo nos dois conjuntos |
| Undersampling | NÃO | a avaliação tem que ser na distribuição real (desbalanceada) |
| Tokenização / stopwords / lemmatizing | SIM | mesmo pré-processamento de texto |
| Embedding | SIM *(modelo de treino, sem re-treinar)* | re-treinar seria vazamento e mudaria o espaço vetorial |

### 2. Limpeza de Dados
##### 2.1. Tratamento de Nulos

In [52]:
print(X_test.shape)
X_test['review_text'].isnull().sum()

(25498, 7)


np.int64(0)

In [53]:
# remove nulos
X_test = X_test.dropna(subset=['review_text'])

# paridade de índices
y_test = y_test.loc[X_test.index]

In [54]:
print(X_test.shape, y_test.shape)

(25498, 7) (25498,)


### 3. Transformação
##### 3.1. Remoção de colunas "inúteis"

In [55]:
X_test = X_test.drop(columns=['original_index', 'review_text_processed', 'review_text_tokenized', 'rating', 'kfold_polarity', 'kfold_rating'])
print(f'colunas: {X_test.columns}. tipo: {type(X_test)}')

colunas: Index(['review_text'], dtype='str'). tipo: <class 'pandas.DataFrame'>


##### 3.2. Tratamento na label

In [56]:
y_test = y_test.map({1.0: 2, 0.0: 0})  # {valor_antigo: valor_atual}
y_test = y_test.fillna(1)

In [57]:
y_test.value_counts()

polarity
2.0    20136
1.0     3342
0.0     2020
Name: count, dtype: int64

##### 3.3. PROCESSAMENTO DE LINGUAGEM NATURAL
##### 3.3.1. Tokenização

In [58]:
X_test['review_text'] = X_test['review_text'].apply(word_tokenize)

##### 3.3.2. Remoção de stopwords e pontuação

In [59]:
X_test['review_text'] = X_test['review_text'].apply(lambda x: remove_stopwords_punctuation(x))

##### 3.3.3. Lemmatizing

Reaproveita o `nlp` (`pt_core_news_sm`) já carregado na seção de treino.

In [60]:
X_test["review_text"] = X_test["review_text"].apply(lambda x: lematiza_tokens(x, nlp))

##### 3.3.4. Embedding

Usa `model_embedding` **treinado no conjunto de treino** — treinar um novo modelo seria vazamento de dados e resultaria em um espaço vetorial incompatível com o do treino.

In [61]:
X_test['review_text'] = X_test['review_text'].apply(lambda x: embedding_ft(x, model_embedding))
X_test = X_test.rename(columns={'review_text': 'review_embedding'})

In [62]:
X_test

,review_embedding
36053,"[[0.483037, 0.9771126, 0.9438422, 1.2974225, 0..."
55549,"[[-0.29945263, 1.016801, 1.0716661, 1.5223943,..."
51600,"[[1.5613955, 1.0624086, 0.9276398, 0.7517864, ..."
12053,"[[0.33228523, 0.8644066, 0.40887916, 0.7447809..."
59505,"[[0.1723928, 0.42492554, 0.19675086, 0.3769994..."
...,...
45623,"[[-0.583551, 0.78032035, 1.3553377, 2.1414065,..."
48331,"[[0.5813414, 1.375777, 0.9389952, 1.5162888, 0..."
68600,"[[0.5618109, 0.8221897, 0.8423225, 1.0907425, ..."
6365,"[[-0.29945263, 1.016801, 1.0716661, 1.5223943,..."


### 4. Salvando os dados

In [63]:
salva_embeddings(X_test, y_test, 'data/X_test.pt', 'data/y_test.pt', vector_size=100)